1. Introducción y Selección de Sujetos:

Para esta práctica se seleccionaron dos participantes del repositorio abierto ds005366: sub-002 y sub-004, enfocándonos en la tarea motora de manos y dedos (Motor2Class).   

La selección de estos sujetos consideró las advertencias técnicas del dataset:   

    Se descartaron participantes con lesiones o resecciones tumorales.   
    Se evitaron sujetos que presentaban más de un 10% de volúmenes atípicos (outliers) o desplazamientos de cabeza superiores a 4 mm.   
    Se contempló que el sujeto sub-002 cuenta con escaneo estructural en 1.5 Teslas, mientras que sub-004 fue adquirido en 3 Teslas, lo que permitió evaluar el comportamiento del pipeline ante diferentes intensidades de campo magnético y niveles de ruido.   

In [ ]:
import os
import glob
import time
import ants
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from nilearn import datasets

# Configuración de rutas locales
CARPETA_DATASET = r"C:\Users\josue_7e6wc9v\Documents\Seguimiento_PDI\ds005366"
DIR_SALIDA = r"C:\Users\josue_7e6wc9v\Documents\Seguimiento_PDI\resultados"
SUJETOS = ["sub-002", "sub-004"]

os.makedirs(DIR_SALIDA, exist_ok=True)

# 1. Función para cargar la plantilla MNI152 (2 mm)
def obtener_plantilla_mni(dir_salida):
    ruta_mni = os.path.join(dir_salida, "mni152_template_2mm.nii.gz")
    if not os.path.exists(ruta_mni):
        print("Descargando plantilla MNI152 (2 mm)...")
        mni_nii = datasets.load_mni152_template(resolution=2)
        mni_nii.to_filename(ruta_mni)
    return ants.image_read(ruta_mni)

# 2. Función para localizar automáticamente los archivos BIDS válidos (> 1 MB)
def encontrar_archivos_bids(raiz_dataset, sujeto_id):
    dir_sujeto = os.path.join(raiz_dataset, sujeto_id)
    if not os.path.exists(dir_sujeto):
        coincidencias = glob.glob(os.path.join(raiz_dataset, "**", sujeto_id), recursive=True)
        if coincidencias:
            dir_sujeto = coincidencias[0]
        else:
            return None

    candidatos_bold = glob.glob(os.path.join(dir_sujeto, "**", "func", "*Motor2Class*bold.nii*"), recursive=True)
    if not candidatos_bold:
        candidatos_bold = glob.glob(os.path.join(dir_sujeto, "**", "func", "*bold.nii*"), recursive=True)
        
    bold_path = next((c for c in candidatos_bold if os.path.getsize(c) > 1024 * 1024), None)

    candidatos_t1 = glob.glob(os.path.join(dir_sujeto, "**", "anat", "*T1w.nii*"), recursive=True)
    t1w_path = next((c for c in candidatos_t1 if os.path.getsize(c) > 1024 * 1024), None)

    if bold_path and t1w_path:
        return {"id": sujeto_id, "bold": bold_path, "t1w": t1w_path}
    return None

# 3. Función para visualizar en el notebook los 3 planos (Sagital, Coronal, Axial)
def mostrar_cortes_3d(img_ants, titulo, img_referencia_contorno=None, cmap="gray"):
    """
    Si la imagen es 4D (serie temporal), extrae el primer volumen para mostrarlo.
    Opcionalmente superpone los contornos en rojo de una imagen de referencia (T1w o MNI).
    """
    datos = img_ants.numpy()
    if datos.ndim == 4:
        datos = datos[:, :, :, 0]  # Primer volumen temporal
        
    nx, ny, nz = datos.shape
    cx, cy, cz = nx // 2, ny // 2, nz // 2
    
    cortes = [datos[cx, :, :].T, datos[:, cy, :].T, datos[:, :, cz].T]
    nombres_planos = [f"Sagital (x={cx})", f"Coronal (y={cy})", f"Axial (z={cz})"]
    
    if img_referencia_contorno is not None:
        ref = img_referencia_contorno.numpy()
        cortes_ref = [ref[cx, :, :].T, ref[:, cy, :].T, ref[:, :, cz].T]
    
    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    fig.suptitle(titulo, fontsize=13, fontweight="bold")
    
    for i in range(3):
        axes[i].imshow(cortes[i], cmap=cmap, origin="lower")
        if img_referencia_contorno is not None:
            axes[i].contour(cortes_ref[i], levels=3, colors="cyan", linewidths=0.7)
        axes[i].set_title(nombres_planos[i], fontsize=10)
        axes[i].axis("off")
        
    plt.tight_layout()
    plt.show()

# Cargar plantilla MNI y su máscara cerebral
mni_template = obtener_plantilla_mni(DIR_SALIDA)
mni_mask = ants.get_mask(mni_template)
print("Entorno configurado y plantilla MNI152 lista.")

## Carga de imágenes crudas (BOLD y T1w)
Se leen desde el disco las imágenes originales de los dos sujetos y se grafican para verificar su estado inicial.

* **`ants.image_read(filename)`**:
  * **Qué ingresa:** `filename` (ruta en texto del archivo `.nii` o `.nii.gz`).
  * **Qué hace:** Lee el archivo NIfTI y devuelve un objeto `ANTsImage` que conserva la matriz de vóxeles (3D para T1w, 4D para BOLD) y sus coordenadas físicas en milímetros.

In [ ]:
# Diccionario para almacenar los datos en memoria a lo largo de las celdas
datos_sujetos = {}


for s_id in SUJETOS:
    info = encontrar_archivos_bids(CARPETA_DATASET, s_id)
    if info:
        print(f"Cargando imágenes de {s_id}...")
        bold_raw = ants.image_read(info["bold"])
        t1w_raw = ants.image_read(info["t1w"])
        
        datos_sujetos[s_id] = {
            "bold_raw": bold_raw,
            "t1w_raw": t1w_raw,
            "ruta_bold": info["bold"],
            "ruta_t1w": info["t1w"]
        }
        
        # Mostrar T1w estructural cruda
        mostrar_cortes_3d(
            t1w_raw, 
            titulo=f"{s_id} | Imagen Anatómica T1w Cruda - Dim: {t1w_raw.shape}"
        )
        # Mostrar primer volumen de la serie BOLD funcional cruda
        mostrar_cortes_3d(
            bold_raw, 
            titulo=f"{s_id} | Imagen Funcional BOLD Cruda (Volumen 0) - Dim: {bold_raw.shape}"
        )
    else:
        print(f"[ERROR] No se encontraron los archivos para {s_id}")

## Paso 1: Realineación (Corrección de movimiento)
Corrige los movimientos de la cabeza del sujeto a lo largo del tiempo alineando todos los volúmenes de la serie funcional contra un volumen promedio mediante una transformación rígida (6 grados de libertad: 3 traslaciones y 3 rotaciones).

* **`ants.get_average_of_timeseries(image)`**:
  * **Qué ingresa:** `image` (la serie funcional 4D `bold_raw`).
  * **Qué hace:** Calcula el promedio de todos los volúmenes en el tiempo y devuelve una imagen 3D (`vol_ref`) que sirve como referencia fija.
* **`ants.motion_correction(image, fixed, type_of_transform)`**:
  * **Qué ingresa:** `image` (serie funcional 4D a corregir), `fixed` (volumen 3D de referencia `vol_ref`), `type_of_transform="BOLDRigid"` (modelo rígido de 6 parámetros).
  * **Qué hace:** Rota y traslada cada volumen de la serie 4D para alinearlo con la referencia. Devuelve un diccionario del cual extraemos `["motion_corrected"]`.
* **`ants.image_write(image, filename)`**:
  * **Qué ingresa:** `image` (objeto `ANTsImage`) y `filename` (ruta de salida).
  * **Qué hace:** Guarda la imagen procesada en el disco duro en formato `.nii.gz`.

In [ ]:
for s_id, d in datos_sujetos.items():
    print(f" [Paso 1/4] Ejecutando Realineación (MoCo) para {s_id}...")
    
    bold_raw = d["bold_raw"]
    
    # Volumen de referencia (promedio de la serie temporal inicial)
    vol_ref = ants.get_average_of_timeseries(bold_raw)
    
    # Registro rígido (6 GDL) de cada volumen hacia la referencia
    moco = ants.motion_correction(
        image=bold_raw, 
        fixed=vol_ref, 
        type_of_transform="BOLDRigid"
    )
    
    bold_realigned = moco["motion_corrected"]
    mean_bold = ants.get_average_of_timeseries(bold_realigned)
    
    # Guardar en el diccionario y en disco
    d["bold_realigned"] = bold_realigned
    d["mean_bold"] = mean_bold
    ants.image_write(bold_realigned, os.path.join(DIR_SALIDA, f"{s_id}_1_realigned.nii.gz"))
    ants.image_write(mean_bold, os.path.join(DIR_SALIDA, f"{s_id}_1_mean_bold.nii.gz"))
    
    # Visualización del resultado de la Realineación
    mostrar_cortes_3d(
        mean_bold, 
        titulo=f"{s_id} | Paso 1: Volumen Funcional Medio tras Realineación (BOLDRigid)"
    )

## Paso 2: Corregistro (Funcional a Anatómica T1w)
Alinea el volumen funcional promedio (baja resolución espacial) con la imagen anatómica T1w de alta resolución del mismo paciente. Como tienen contrastes de color opuestos, utiliza Información Mutua.

* **`ants.n4_bias_field_correction(image)`**:
  * **Qué ingresa:** `image` (la imagen anatómica `t1w_raw`).
  * **Qué hace:** Corrige las variaciones de brillo causadas por la bobina del resonador, dejando homogénea la intensidad de los tejidos.
* **`ants.registration(fixed, moving, type_of_transform, aff_metric, aff_sampling)`**:
  * **Qué ingresa:** `fixed` (imagen fija `t1w_n4`), `moving` (imagen móvil `mean_bold`), `type_of_transform="Rigid"` (transformación rígida), `aff_metric="mattes"` (métrica de Información Mutua) y `aff_sampling=32` (número de bins del histograma).
  * **Qué hace:** Calcula la rotación y traslación para encajar el volumen funcional dentro de la anatomía T1w. Devuelve `"warpedmovout"` (imagen alineada) y `"fwdtransforms"` (matriz de transformación).

In [ ]:
for s_id, d in datos_sujetos.items():
    print(f" [Paso 2/4] Ejecutando Corregistro (BOLD -> T1w) para {s_id}...")
    
    t1w_raw = d["t1w_raw"]
    mean_bold = d["mean_bold"]
    
    # 1. Corrección de inhomogeneidad del campo magnético (N4)
    t1w_n4 = ants.n4_bias_field_correction(t1w_raw)
    
    # 2. Registro intermodal rígido con Información Mutua
    tx_coreg = ants.registration(
        fixed=t1w_n4,
        moving=mean_bold,
        type_of_transform="Rigid",
        aff_metric="mattes",
        aff_sampling=32
    )
    
    coreg_mean = tx_coreg["warpedmovout"]
    
    # Guardar en memoria y en disco
    d["t1w_n4"] = t1w_n4
    d["tx_coreg"] = tx_coreg
    d["coreg_mean"] = coreg_mean
    ants.image_write(coreg_mean, os.path.join(DIR_SALIDA, f"{s_id}_2_coreg_mean.nii.gz"))
    
    # Visualización: Imagen funcional corregistrada con contornos de T1w superpuestos en cian
    mostrar_cortes_3d(
        coreg_mean, 
        titulo=f"{s_id} | Paso 2: BOLD Medio Corregistrado (Contornos Cian = Anatomía T1w)",
        img_referencia_contorno=t1w_n4
    )

## Paso 3: Normalización al espacio MNI152 y pruebas de parámetros
Deforma el cerebro de cada sujeto hacia la plantilla estándar MNI152 para permitir comparaciones grupales. Se ponen a prueba 3 configuraciones variando los parámetros de `ants.registration`:

* **`ants.registration(fixed, moving, type_of_transform, syn_metric, syn_sampling, reg_iterations, grad_step, flow_sigma)`**:
  * **Qué ingresa y qué mejora la normalización:**
    * `fixed` y `moving`: Plantilla `mni_template` e imagen del sujeto `t1w_n4`.
    * `type_of_transform`: Pasar de `"Affine"` (solo escala y rota globalmente con 12 parámetros) a `"SyN"` (aplica deformación local no lineal para ajustar surcos y ventrículos).
    * `syn_metric` y `syn_sampling`: Cambiar `"mattes"` (Información Mutua global) por `"CC"` (Correlación Cruzada Local) con radio `syn_sampling=4` vóxeles, lo que mejora el ajuste de bordes entre sustancia gris y blanca en imágenes de la misma modalidad (T1w a T1w).
    * `reg_iterations`: Pasar de `(40, 20, 0)` (donde el 0 final omite iterar a resolución real) a `(100, 70, 50, 20)` obliga al optimizador a refinar 20 iteraciones en la resolución nativa del vóxel.
    * `grad_step=0.1` y `flow_sigma=3`: Controlan el tamaño de paso del optimizador y suavizan el campo de deformación para evitar estiramientos bruscos.
* **`ants.image_similarity(fixed, moving, metric_type, fixed_mask)`**:
  * **Qué ingresa:** La plantilla `mni_template`, la imagen normalizada `warped`, el tipo de métrica (`"MattesMutualInformation"` o `"Correlation"`) y la máscara `mni_mask`.
  * **Qué hace:** Mide numéricamente qué tan parecidas quedaron las imágenes (valores más negativos indican mejor alineación).
* **`ants.apply_transforms(fixed, moving, transformlist, imagetype, interpolator)`**:
  * **Qué ingresa:** `fixed` (`mni_template`), `moving` (`bold_realigned`), `transformlist` (unión de las transformaciones de normalización y corregistro), `imagetype=3` (indica serie 4D) e `interpolator="linear"`.
  * **Qué hace:** Aplica todas las transformaciones juntas en un solo paso sobre la serie funcional 4D para llevarla al espacio MNI sin interpolar dos veces.

In [ ]:
configuraciones_norm = [
    {
        "nombre": "1_Lineal_Affine",
        "desc": "Solo transformación lineal global (12 GDL)",
        "params": {"type_of_transform": "Affine", "aff_metric": "mattes"}
    },
    {
        "nombre": "2_SyN_Default_MI",
        "desc": "Difeomórfico rápido (sin refinamiento en resolución nativa)",
        "params": {
            "type_of_transform": "SyN", "syn_metric": "mattes",
            "syn_sampling": 32, "reg_iterations": (40, 20, 0), "grad_step": 0.2
        }
    },
    {
        "nombre": "3_SyN_Optimizado_CC",
        "desc": "SyN + Cross-Correlation local (r=4) + 4 niveles de pirámide",
        "params": {
            "type_of_transform": "SyN", "syn_metric": "CC",
            "syn_sampling": 4, "reg_iterations": (100, 70, 50, 20),
            "grad_step": 0.1, "flow_sigma": 3
        }
    }
]

todas_las_metricas = []

for s_id, d in datos_sujetos.items():
    print(f" [Paso 3/4] Pruebas y Normalización a MNI152 para {s_id}...")
    
    t1w_n4 = d["t1w_n4"]
    imagenes_warped = {}
    tx_optima = None
    resultados_sujeto = []
    
    for cfg in configuraciones_norm:
        print(f" -> Probando: {cfg['nombre']}...")
        t0 = time.time()
        tx = ants.registration(fixed=mni_template, moving=t1w_n4, **cfg["params"])
        t_total = time.time() - t0
        
        warped = tx["warpedmovout"]
        imagenes_warped[cfg["nombre"]] = warped
        
        if cfg["nombre"] == "3_SyN_Optimizado_CC":
            tx_optima = tx
            
        sim_mi = ants.image_similarity(mni_template, warped, metric_type="MattesMutualInformation", fixed_mask=mni_mask)
        sim_cc = ants.image_similarity(mni_template, warped, metric_type="Correlation", fixed_mask=mni_mask)
        
        fila = {
            "Sujeto": s_id,
            "Configuración": cfg["nombre"],
            "Descripción": cfg["desc"],
            "Tiempo (s)": round(t_total, 2),
            "Mattes MI (más neg = mejor)": round(sim_mi, 4),
            "Correlación (más neg = mejor)": round(sim_cc, 4)
        }
        resultados_sujeto.append(fila)
        todas_las_metricas.append(fila)
        
    # 1. Visualizar en el notebook la comparación de las 3 configuraciones de normalización T1w
    z_slice = mni_template.shape[2] // 2
    fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
    fig.suptitle(f"Comparación de Normalización T1w hacia MNI152 - {s_id}", fontsize=14, fontweight="bold")
    
    axes[0].imshow(mni_template.numpy()[:, :, z_slice].T, cmap="gray", origin="lower")
    axes[0].set_title("Referencia MNI152", fontsize=11, fontweight="bold")
    axes[0].axis("off")
    
    for idx, cfg in enumerate(configuraciones_norm):
        nombre = cfg["nombre"]
        img_np = imagenes_warped[nombre].numpy()[:, :, z_slice].T
        axes[idx + 1].imshow(img_np, cmap="gray", origin="lower")
        axes[idx + 1].contour(mni_template.numpy()[:, :, z_slice].T, levels=3, colors="red", linewidths=0.7)
        axes[idx + 1].set_title(
            f"{nombre}\nCC: {resultados_sujeto[idx]['Correlación (más neg = mejor)']:.4f} | {resultados_sujeto[idx]['Tiempo (s)']}s",
            fontsize=10
        )
        axes[idx + 1].axis("off")
        
    plt.tight_layout()
    plt.savefig(os.path.join(DIR_SALIDA, f"comparacion_normalizacion_{s_id}.png"), dpi=300)
    plt.show()
    
    # 2. Aplicar la transformación óptima (SyN + CC) concatenada con el corregistro a la serie BOLD 4D
    print(f" -> Transformando serie funcional 4D de {s_id} al espacio MNI152...")
    lista_tx = tx_optima["fwdtransforms"] + d["tx_coreg"]["fwdtransforms"]
    bold_mni = ants.apply_transforms(
        fixed=mni_template,
        moving=d["bold_realigned"],
        transformlist=lista_tx,
        imagetype=3,
        interpolator="linear"
    )
    
    d["bold_mni"] = bold_mni
    ants.image_write(tx_optima["warpedmovout"], os.path.join(DIR_SALIDA, f"{s_id}_3_T1w_MNI.nii.gz"))
    ants.image_write(bold_mni, os.path.join(DIR_SALIDA, f"{s_id}_3_bold_normalized.nii.gz"))
    
    # 3. Visualizar el BOLD normalizado con los contornos de la plantilla MNI152 en cian
    mean_bold_mni = ants.get_average_of_timeseries(bold_mni)
    mostrar_cortes_3d(
        mean_bold_mni,
        titulo=f"{s_id} | Paso 3: BOLD Normalizado en Espacio MNI152 (Contornos Cian = Atlas MNI)",
        img_referencia_contorno=mni_template
    )

# Mostrar tabla resumen de métricas
df_metricas = pd.DataFrame(todas_las_metricas)
df_metricas.to_csv(os.path.join(DIR_SALIDA, "tabla_metricas_normalizacion.csv"), index=False)
display(df_metricas)

Tiempo (s): El costo computacional que le tomó al procesador calcular la deformación.   

Mattes MI y Correlación: Ambas son métricas de costo que miden qué tan bien encaja el cerebro del paciente con la plantilla MNI152. En la librería ANTsPy, el optimizador busca minimizar estas funciones, por lo que entre más negativo sea el número, mayor es la similitud y mejor quedó el registro.

1_Lineal_Affine (Rápida pero insuficiente):Tarda menos de 3 segundos por sujeto. Sus valores de similitud son los peores (los más cercanos a cero, ej. correlación de $-0.0193$ en sub-002 y $-0.0583$ en sub-004).   
 Razón: Solo rota, traslada y escala globalmente el cerebro. No deforma el tejido internamente, por lo que los surcos y ventrículos quedan desfasados. 

2_SyN_Default_MI (Deformable rápida):Tarda entre 6 y 8 segundos.  
 Mejora sustancialmente la Información Mutua (alcanza $-0.2433$ en sub-004).  
  Razón: Aplica deformación no lineal simétrica (SyN). Sin embargo, al usar la configuración por defecto con 0 iteraciones a escala nativa, el algoritmo solo acomoda las estructuras grandes y no refina los detalles pequeños de la corteza. 

3_SyN_Optimizado_CC (Deformable completa):Tarda más tiempo (cerca de 3.2 a 3.5 minutos por sujeto: 193.44 s y 208.90 s).   Logra los mejores puntajes de correlación en ambos cerebros: $-0.0725$ en sub-002 y $-0.2499$ en sub-004.  
 Razón: Al evaluar la correlación en ventanas locales pequeñas ($r=4$) y obligar al optimizador a calcular 20 iteraciones finales en la resolución real del vóxel, los pliegues de la corteza y los ventrículos se ajustan con precisión milimétrica al molde estándar.  


¿Cuál configuración resulta mejor y por qué? 

La mejor configuración anatómica y funcional es la 3_SyN_Optimizado_CC.   

El porqué técnico: En un registro entre imágenes de la misma modalidad ($T_{1w}$ del sujeto hacia el $T_{1w}$ de la plantilla MNI), la Correlación Cruzada Local (CC) es superior a la Información Mutua global (mattes), ya que evalúa la frontera entre sustancia gris y blanca vecindad por vecindad. Sumado a esto, las iteraciones a resolución completa permiten que los surcos cerebrales encajen donde deben.   

El tiempo (trade-off): Aunque pasa de tardar ~7 segundos a ~3.5 minutos, este paso se ejecuta una sola vez sobre la imagen anatómica de cada paciente. En resonancia funcional motora, ese costo en tiempo se justifica plenamente para asegurar que la activación de los dedos o manos quede ubicada con exactitud en la corteza motora primaria

## Paso 4: Suavizado espacial (Smoothing)
Filtra espacialmente la serie funcional normalizada con un kernel Gaussiano pasa-bajas para reducir el ruido de alta frecuencia, mejorar la relación señal-ruido (SNR) y suavizar diferencias anatómicas residuales entre sujetos.

* **`ants.smooth_image(image, sigma, sigma_in_physical_coordinates, FWHM)`**:
  * **Qué ingresa:** `image` (serie 4D `bold_mni`), `sigma=(6.0, 6.0, 6.0, 0.0)` (suavizado de 6 mm en los ejes espaciales X, Y, Z y 0.0 en el eje del tiempo para no mezclar volúmenes temporales), `sigma_in_physical_coordinates=True` (indica que las unidades son milímetros reales) y `FWHM=True` (define el ancho del filtro como *Full Width at Half Maximum*).
  * **Qué hace:** Aplica convolución Gaussiana 3D a cada volumen de la serie temporal y devuelve la imagen funcional suavizada (`bold_smoothed`).

In [ ]:
for s_id, d in datos_sujetos.items():
    print(f" [Paso 4/4] Ejecutando Suavizado Espacial (FWHM = 6 mm) para {s_id}...")
    
    bold_mni = d["bold_mni"]
    
    # Convolución Gaussiana 3D (6 mm en X, Y, Z y 0.0 en tiempo)
    bold_smoothed = ants.smooth_image(
        image=bold_mni,
        sigma=(6.0, 6.0, 6.0, 0.0),
        sigma_in_physical_coordinates=True,
        FWHM=True
    )
    
    d["bold_smoothed"] = bold_smoothed
    ants.image_write(bold_smoothed, os.path.join(DIR_SALIDA, f"{s_id}_4_bold_smoothed.nii.gz"))
    
    # Visualización final del volumen funcional suavizado en los 3 planos
    mostrar_cortes_3d(
        bold_smoothed,
        titulo=f"{s_id} | Paso 4: BOLD Normalizado y Suavizado (Filtro Gaussiano FWHM = 6 mm)"
    )

print("\n>>> Pipeline completado exitosamente para ambos sujetos.")

In [ ]:
for s_id, d in datos_sujetos.items():
    etapas = [
        ("1. BOLD Crudo", d["bold_raw"].numpy()[:, :, :, 0]),
        ("2. Realineado (Media)", d["mean_bold"].numpy()),
        ("3. Corregistrado a T1w", d["coreg_mean"].numpy()),
        ("4. Normalizado a MNI", d["bold_mni"].numpy()[:, :, :, 0]),
        ("5. Suavizado (6 mm)", d["bold_smoothed"].numpy()[:, :, :, 0])
    ]
    
    fig, axes = plt.subplots(1, 5, figsize=(18, 4))
    fig.suptitle(f"Resumen de la Evolución del Preprocesamiento fMRI - {s_id}", fontsize=14, fontweight="bold")
    
    for idx, (nombre_etapa, vol) in enumerate(etapas):
        z_central = vol.shape[2] // 2
        axes[idx].imshow(vol[:, :, z_central].T, cmap="gray", origin="lower")
        axes[idx].set_title(nombre_etapa, fontsize=10, fontweight="bold")
        axes[idx].axis("off")
        
    plt.tight_layout()
    plt.savefig(os.path.join(DIR_SALIDA, f"evolucion_completa_{s_id}.png"), dpi=300)
    plt.show()

Paso 1: Corrección del Tiempo de Corte (Slice Timing) — Omitido

Fundamento teórico: En resonancia funcional estándar, los cortes 2D de un volumen se adquieren sucesivamente durante el tiempo de repetición ($TR$), produciendo un desfase temporal en el muestreo de la curva hemodinámica (HRF) entre diferentes rebanadas del cerebro. Para corregirlo, habitualmente se interpola temporalmente la señal tomando como ancla el corte adquirido a la mitad del ciclo ($t_{\text{ref}} = TR / 2$).   


Paso 2: Realineación (Motion Correction)Fundamento teórico: 

Los movimientos de la cabeza dentro del resonador generan artefactos y falsos positivos. Dado que el cráneo rígido impide que el cerebro cambie de forma entre adquisiciones, se aplica una transformación global rígida de 6 grados de libertad (GDL): 3 parámetros de traslación ($T_x, T_y, T_z$) y 3 de rotación ($\theta_x, \theta_y, \theta_z$).   

Rutina en ANTsPy: Primero se genera el volumen funcional promedio con ants.get_average_of_timeseries() para utilizarlo como imagen fija de referencia. Posteriormente, se ejecuta ants.motion_correction(image=bold, fixed=ref, type_of_transform="BOLDRigid"), la cual alinea cada volumen temporal con la media y calcula el desplazamiento cuadro a cuadro (Framewise Displacement). 

Paso 3: Corregistro Funcional-EstructuralFundamento teórico: 

Las imágenes funcionales BOLD ($T_2^*$) ofrecen buena información dinámica pero baja resolución espacial y pobre contraste anatómico. Para ubicarlas con precisión anatómica, se deben alinear con la imagen estructural $T_{1w}$ de alta resolución del mismo paciente. Dado que son modalidades distintas con contrastes invertidos (el líquido cefalorraquídeo es brillante en $T_2^*$ y oscuro en $T_{1w}$), no se puede usar correlación de intensidades directa. Se emplea una métrica basada en teoría de la información: la Información Mutua (MI), que evalúa la dependencia entre los histogramas de ambas imágenes.   

Rutina en ANTsPy: Se procesó la imagen $T_{1w}$ con ants.n4_bias_field_correction() para eliminar gradientes de baja frecuencia del campo magnético. Luego, se aplicó un registro rígido intermodal con ants.registration(fixed=t1w, moving=mean_bold, type_of_transform="Rigid", aff_metric="mattes", aff_sampling=32).   

Paso 4: Normalización Espacial

Fundamento teórico: Para realizar estudios grupales o comparar cerebros entre diferentes personas, es indispensable deformar la anatomía de cada sujeto hacia un espacio estereotáxico de referencia común (plantilla estándar MNI152). Como los pliegues y surcos de la corteza varían ampliamente de un individuo a otro, una transformación lineal no basta; se requiere una transformación local deformable que mapee los tejidos mediante un campo continuo de deformaciones sin alterar la topología cerebral.   

Rutina en ANTsPy: Se empleó el algoritmo difeomórfico simétrico SyN (ants.registration(fixed=mni, moving=t1w, type_of_transform="SyN")). Para evitar degradar la serie funcional BOLD con múltiples interpolaciones, se concatenaron la matriz de corregistro y el campo no lineal de normalización en un único paso con ants.apply_transforms(fixed=mni, moving=bold_realigned, transformlist=lista_transforms, imagetype=3).   Paso 5: Suavizado Espa

Paso 5: Suavizado Espacial (Spatial Smoothing)

Fundamento teórico: Consiste en la convolución espacial de la imagen tridimensional con un kernel Gaussiano pasa-bajas. Este paso cumple tres objetivos:   Reduce el ruido de alta frecuencia e incrementa la relación señal-ruido (SNR) en cúmulos de activación funcionales que abarcan varios vóxeles contiguos.   Compensa la variabilidad anatómica residual entre sujetos que la normalización no logra corregir por completo.   Asegura que los datos cumplan los supuestos de continuidad espacial requeridos por la Teoría de Campos Aleatorios Gaussianos para la corrección estadística de comparaciones múltiples.   

Rutina en ANTsPy: Se utilizó ants.smooth_image(image=bold_mni, sigma=(6.0, 6.0, 6.0, 0.0), FWHM=True). Se fijó un kernel isotrópico de 6 mm FWHM en los ejes espaciales (acorde al tamaño de 2 a 3 veces el vóxel y validado en la literatura de este dataset) y se mantuvo un valor de 0 en el cuarto eje para no mezclar la información a lo largo de la dimensión temporal.

3. Parámetros que Mejoran la Normalización Espacial

A partir del esquema general de registro (medida de similitud, optimizador, transformación e interpolador), se identificaron y evaluaron los siguientes parámetros en ants.registration() para mejorar el ajuste hacia el atlas MNI152:

   Modelo de transformación (type_of_transform): Pasar de un modelo global lineal ("Affine", 12 parámetros de traslación, rotación, escala y torsión) al modelo deformable difeomórfico ("SyN"). Mientras que el modelo afín solo ajusta el contorno exterior del encéfalo, SyN aplica una etapa global inicial seguida de deformaciones locales suaves que logran emparejar surcos y ventrículos.   
   
   Métrica de similitud no lineal (syn_metric y syn_sampling): En lugar de usar Información Mutua ("mattes"), que evalúa histogramas globales y es más apropiada para imágenes de distinta modalidad, en la normalización ($T_{1w}$ individual a $T_{1w}$ MNI152) resulta superior usar Correlación Cruzada Local (syn_metric="CC") con un radio de vecindad de 4 vóxeles (syn_sampling=4). Esto permite comparar el contraste local ventana por ventana, adaptándose mejor a los bordes entre sustancia gris y blanca.   
   
   Niveles de la pirámide multirresolución (reg_iterations): Las rutinas rápidas por defecto utilizan esquemas como (40, 20, 0), donde el último valor en cero indica que se omite el refinamiento en la resolución nativa del vóxel para ahorrar tiempo. Al cambiar este parámetro a (100, 70, 50, 20), el optimizador ejecuta 20 iteraciones finales a resolución completa ($1\times$), logrando un ajuste fino de los pliegues corticales.   
   
   Regularización y tamaño de paso (grad_step y flow_sigma): Disminuir el paso de actualización del optimizador a grad_step=0.1 y regularizar el campo de deformación con flow_sigma=3 controla la elasticidad del mapeo y evita deformaciones bruscas.   